<a href="https://colab.research.google.com/github/igorfantucci/Aula-Automatica---GRUPO-5/blob/main/etapa-01-logica/09%20-%20Motor%20de%20Inferencia%20Forward%20e%20Backward%20Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 09 - Notebook: Motores de Inferência Forward e Backward Chaining
## Processo: Planta Industrial de Produção de Biodiesel (Transesterificação em Batelada - Grupo 5)

---

### 1. Fundamentos Matemáticos e Algorítmicos: Inferência em Sistemas Baseados em Regras

Um **Sistema Especialista Baseado em Regras (Rule-Based Expert System)** no contexto do SCADA-Core da planta de biodiesel opera sobre uma tríade formal $(\mathcal{F}, \mathcal{R}, \mathcal{M})$:
1. **Base de Fatos Dinâmica ($\mathcal{F}$):** Conjunto de proposições atômicas que representam a telemetria em tempo real dos sensores (ISA-5.1) e as variáveis de processo discretizadas ($p_1, t_{\text{alta}}, g_{\text{alm}}, v_{\text{in\_mix}}, \dots$).
2. **Base de Conhecimento Especialista ($\mathcal{R}$):** Conjunto de regras formais na forma de Cláusulas de Horn definidas como:
   $$R_i: \quad \left( \bigwedge_{j=1}^{k} A_{i,j} \right) \longrightarrow C_i$$
3. **Motor de Inferência ($\mathcal{M}$):** Algoritmo responsável pelo raciocínio automatizado, dividindo-se em duas abordagens canônicas:

* **Encadeamento para Frente (*Forward Chaining* — Data-Driven / Bottom-Up):**
  * **Princípio:** Inicia a partir dos **fatos conhecidos** (leitura dos sensores) e dispara sucessivamente as regras cujos antecedentes são satisfeitos via aplicação direta de *Modus Ponens*:
    $$\frac{A_{i,1} \land A_{i,2} \land \dots \land A_{i,k}, \quad \left( \bigwedge_{j=1}^{k} A_{i,j} \rightarrow C_i \right)}{C_i}$$
  * **Resolução de Conflitos:** Ordenação da agenda por **Prioridade de Segurança SIL** (10 = Crítica, $t_{\text{max}} \le 0.5\text{ s}$) e **Especificidade** de antecedentes, executando até alcançar o ponto fixo (*Fixed Point*).

* **Encadeamento para Trás (*Backward Chaining* — Goal-Driven / Top-Down):**
  * **Princípio:** Inicia a partir de uma **meta ou hipótese** a ser investigada (ex.: *"O reator R-200 está sob risco de runaway térmico?"* ou *"A válvula de metóxido XV-202 deve ser desarmada?"*) e busca recursivamente provar as submetas necessárias.
  * **Explicabilidade e Auditoria Forense (*XAI*):** Gera automaticamente a **Árvore de Justificativa Dedutiva (*Proof Tree / Audit Trail*)**, rastreando toda a cadeia causal de falhas para auditoria pós-incidente conforme as normas IEC 61508 / IEC 61511.

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from dataclasses import dataclass, field
from typing import Set, Tuple, List, Dict, Optional, Any
import time

@dataclass
class Fato:
    nome: str
    valor: bool
    descricao: str
    fonte: str = "SENSOR"  # 'SENSOR' ou 'INFERIDO'
    timestamp: float = field(default_factory=time.time)

@dataclass
class RegraDiagnostico:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    severidade: str       # 'CRÍTICA', 'ALTA', 'MÉDIA', 'BAIXA'
    prioridade: int       # 1 a 10 (10 = mais urgente)
    tempo_resposta_max_s: float
    procedimento_pop: str

print("[OK] Estruturas de dados Fato e RegraDiagnostico carregadas com sucesso!")


[OK] Estruturas de dados Fato e RegraDiagnostico carregadas com sucesso!


---
## 2. Modelagem Orientada a Objetos da Base de Conhecimento e do Motor Especialista

In [2]:
class BaseConhecimentoSCADA:
    def __init__(self):
        self.regras: List[RegraDiagnostico] = []
        self._indice_antecedentes: Dict[str, List[RegraDiagnostico]] = {}
        self._indice_consequentes: Dict[str, List[RegraDiagnostico]] = {}

    def adicionar_regra(
        self, id_regra: str, antecedentes: List[str], consequente: str,
        descricao: str, severidade: str = "ALTA", prioridade: int = 5,
        tempo_max_s: float = 5.0, pop: str = "Verificar malha operacional"
    ):
        regra = RegraDiagnostico(
            id_regra=id_regra,
            antecedentes=set(antecedentes),
            consequente=consequente,
            descricao_diagnostico=descricao,
            severidade=severidade,
            prioridade=prioridade,
            tempo_resposta_max_s=tempo_max_s,
            procedimento_pop=pop
        )
        self.regras.append(regra)

        # Indexação bidirecional para buscas em O(1)
        for ant in antecedentes:
            if ant not in self._indice_antecedentes:
                self._indice_antecedentes[ant] = []
            self._indice_antecedentes[ant].append(regra)

        if consequente not in self._indice_consequentes:
            self._indice_consequentes[consequente] = []
        self._indice_consequentes[consequente].append(regra)

    def obter_regras_por_fato(self, fato_nome: str) -> List[RegraDiagnostico]:
        return self._indice_antecedentes.get(fato_nome, [])

    def obter_regras_por_consequente(self, meta: str) -> List[RegraDiagnostico]:
        return self._indice_consequentes.get(meta, [])

    def exportar_catalogo(self) -> List[Dict[str, Any]]:
        catalogo = []
        for r in sorted(self.regras, key=lambda x: (x.prioridade, len(x.antecedentes)), reverse=True):
            catalogo.append({
                "ID": r.id_regra,
                "Prioridade": r.prioridade,
                "Severidade": r.severidade,
                "SE (Antecedentes)": " AND ".join(sorted(r.antecedentes)),
                "ENTÃO (Consequente)": r.consequente,
                "Diagnóstico": r.descricao_diagnostico,
                "POP": r.procedimento_pop
            })
        return catalogo

class MotorInferencia:
    def __init__(self, base_conhecimento: BaseConhecimentoSCADA):
        self.bc = base_conhecimento

    def forward_chaining(self, fatos_iniciais: Set[str]) -> Tuple[Set[str], List[Dict[str, Any]]]:
        """
        Algoritmo Forward Chaining (Data-Driven / Guiado por Dados):
        Aplica Modus Ponens sucessivo com resolução de conflitos por prioridade SIL
        e especificidade de antecedentes até alcançar ponto fixo (Fixed Point).
        """
        fatos_conhecidos = set(fatos_iniciais)
        historico_disparos = []
        passo = 1
        novos_fatos = True

        while novos_fatos:
            novos_fatos = False
            # Resolução de conflitos: Maior Prioridade DESC, Maior Cardinalidade de Antecedentes DESC
            regras_candidatas = sorted(
                self.bc.regras,
                key=lambda r: (r.prioridade, len(r.antecedentes), r.id_regra),
                reverse=True
            )

            for regra in regras_candidatas:
                if regra.antecedentes.issubset(fatos_conhecidos) and regra.consequente not in fatos_conhecidos:
                    fatos_conhecidos.add(regra.consequente)
                    historico_disparos.append({
                        "Passo": passo,
                        "ID Regra": regra.id_regra,
                        "Prioridade": regra.prioridade,
                        "Severidade": regra.severidade,
                        "Fato Inferido": regra.consequente,
                        "Diagnóstico": regra.descricao_diagnostico,
                        "Procedimento POP": regra.procedimento_pop
                    })
                    passo += 1
                    novos_fatos = True
                    break  # Dispara uma regra por ciclo garantindo estrita hierarquia de segurança

        return fatos_conhecidos, historico_disparos

    def backward_chaining(
        self,
        meta: str,
        fatos_iniciais: Set[str],
        visitados: Optional[Set[str]] = None,
        nivel: int = 0
    ) -> Tuple[bool, List[str], Dict[str, Any]]:
        """
        Algoritmo Backward Chaining (Goal-Driven / Guiado por Metas):
        Investigação recursiva de hipóteses com geração de árvore de prova dedutiva (Proof Tree).
        """
        if visitados is None:
            visitados = set()

        # Caso Base: A meta já é um fato comprovado na base de dados
        if meta in fatos_iniciais:
            trilha = [f"{'  ' * nivel}[FATO CONFIRMADO] Hipótese '{meta}' presente na base de fatos."]
            arvore = {"meta": meta, "status": "PROVADO_POR_FATO", "subarvores": []}
            return True, trilha, arvore

        # Prevenção contra dependências circulares / loops
        if meta in visitados:
            trilha = [f"{'  ' * nivel}[LOOP DETECTADO] Meta '{meta}' em recursão cíclica."]
            arvore = {"meta": meta, "status": "FALHA_LOOP", "subarvores": []}
            return False, trilha, arvore

        visitados.add(meta)
        regras_para_meta = self.bc.obter_regras_por_consequente(meta)

        if not regras_para_meta:
            trilha = [f"{'  ' * nivel}[FALHA] Nenhuma regra na base deduz '{meta}' e não é fato conhecido."]
            arvore = {"meta": meta, "status": "FALHA_SEM_REGRAS", "subarvores": []}
            visitados.remove(meta)
            return False, trilha, arvore

        trilha_completa = [f"{'  ' * nivel}[AVALIANDO META] Investigando '{meta}' via {len(regras_para_meta)} regra(s)..."]

        for regra in sorted(regras_para_meta, key=lambda r: r.prioridade, reverse=True):
            trilha_completa.append(f"{'  ' * (nivel+1)}-> Testando Regra [{regra.id_regra}]: SE ({' AND '.join(sorted(regra.antecedentes))}) ENTÃO {regra.consequente}")
            
            todos_antecedentes_provados = True
            subarvores = []

            for ant in sorted(regra.antecedentes):
                provado, sub_trilha, sub_arv = self.backward_chaining(
                    ant, fatos_iniciais, visitados, nivel + 2
                )
                trilha_completa.extend(sub_trilha)
                subarvores.append(sub_arv)
                if not provado:
                    todos_antecedentes_provados = False
                    trilha_completa.append(f"{'  ' * (nivel+2)}x Falha ao provar antecedente '{ant}'. Regra [{regra.id_regra}] descartada.")
                    break

            if todos_antecedentes_provados:
                trilha_completa.append(f"{'  ' * (nivel+1)}[SUCESSO] Regra [{regra.id_regra}] satisfeita! Meta '{meta}' PROVADA com sucesso.")
                arvore = {
                    "meta": meta,
                    "status": "PROVADO_POR_REGRA",
                    "regra": regra.id_regra,
                    "diagnostico": regra.descricao_diagnostico,
                    "pop": regra.procedimento_pop,
                    "subarvores": subarvores
                }
                visitados.remove(meta)
                return True, trilha_completa, arvore

        visitados.remove(meta)
        trilha_completa.append(f"{'  ' * nivel}[FALHA] Todas as tentativas para provar '{meta}' falharam.")
        arvore = {"meta": meta, "status": "FALHA_TODAS_REGRAS", "subarvores": []}
        return False, trilha_completa, arvore

print("[OK] Classes BaseConhecimentoSCADA e MotorInferencia inicializadas com sucesso!")


[OK] Classes BaseConhecimentoSCADA e MotorInferencia inicializadas com sucesso!


---
## 3. Cadastro e Indexação da Base de Conhecimento Especialista da Planta de Biodiesel

A base de conhecimento cobre as anomalias e contingências operacionais nos Setores 100, 200, 300 e 400 da planta de biodiesel (Transesterificação em Batelada):

In [3]:
# Instanciação da Base de Conhecimento
bc = BaseConhecimentoSCADA()

# Regra R-01: Runaway Térmico no Reator R-200
bc.adicionar_regra(
    "R-01", ["p1", "t_alta"], "EXOTERMIA_RUNAWAY_REATOR",
    "Exotermia Descontrolada e Sobrepressão no Reator R-200", "CRÍTICA", 10, 0.5,
    "POP-SIS-01: Desarme total de HT-201, corte de XV-202 e abertura plena de CW-201"
)

# Regra R-02: Corte de Alimentação de Metóxido por Reação Fora de Controle
bc.adicionar_regra(
    "R-02", ["EXOTERMIA_RUNAWAY_REATOR", "v_in_mix"], "TRIP_ALIMENTACAO_METOXIDO",
    "Corte Imediato da Dosagem de Metóxido por Reação Fora de Controle", "CRÍTICA", 10, 0.5,
    "POP-SIS-02: Fechar imediatamente XV-202, desenergizar P-102 e inertizar com N2"
)

# Regra R-03: Fuga de Vapores Inflamáveis de Metanol no Setor 100
bc.adicionar_regra(
    "R-03", ["g_alm"], "VAZAMENTO_GAS_METANOL_S100",
    "Detecção de Vapores Inflamáveis/Tóxicos de Metanol no Setor 100", "CRÍTICA", 9, 1.0,
    "POP-SST-03: Cortar XV-102 e XV-202, ligar exaustão e desenergizar bombas P-102"
)

# Regra R-04: Transbordamento no Reator por Óleo Vegetal
bc.adicionar_regra(
    "R-04", ["l_alto", "v_in_oleo"], "TRANSBORDAMENTO_REATOR_R200",
    "Sobrecarga Volumétrica de Óleo Vegetal no Reator de Transesterificação", "CRÍTICA", 9, 1.0,
    "POP-PR-01: Fechar XV-201, desligar bomba de óleo P-101 e reter batelada"
)

# Regra R-05: Inibição de Aquecimento sem Resfriamento de Emergência Disponível
bc.adicionar_regra(
    "R-05", ["h1", "not_r1"], "OPERACAO_TERMICA_SEM_SALVAGUARDA",
    "Acionamento de Aquecedor HT-201 sem Circuito de Resfriamento de Emergência Disponível", "CRÍTICA", 9, 1.0,
    "POP-SIS-04: Trip imediato de HT-201 e alarme de manutenção na linha CW-201"
)

# Regra R-06: Risco de Cavitação / Operação a Seco do Agitador
bc.adicionar_regra(
    "R-06", ["l_baixo", "m_reator"], "RISCO_CAVITACAO_AGITADOR_R200",
    "Operação do Agitador sem Carga Hidráulica Mínima no Reator R-200", "ALTA", 8, 2.0,
    "POP-MA-05: Desarmar inversor de AG-201 e bloquear aquecimento HT-201"
)

# Regra R-07: Perda de Biodiesel no Dreno de Glicerina
bc.adicionar_regra(
    "R-07", ["v_glic", "not_i_glic"], "PERDA_BIODIESEL_DRENO_GLICERINA",
    "Drenagem Indevida de Biodiesel Bruto pela Linha de Fundo de Glicerina", "ALTA", 8, 1.5,
    "POP-SEP-02: Fechar válvula proporcional XV-301 e reajustar tempo de decantação"
)

# Regra R-08: Transferência Final sem Fluxo de Lavagem e Neutralização
bc.adicionar_regra(
    "R-08", ["b_final", "not_f_lav"], "IMPUREZA_CATALISADOR_BIODIESEL",
    "Transferência de Biodiesel sem Etapa de Lavagem e Neutralização Concluída", "ALTA", 7, 3.0,
    "POP-PUR-04: Bloquear bomba P-401, fechar XV-401 e restabelecer água desmineralizada"
)

print("=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (PLANTA DE BIODIESEL) ===")
print(formatar_tabela(bc.exportar_catalogo()))

assert len(bc.regras) == 8
assert len(bc.obter_regras_por_fato("p1")) >= 1
print("\n[OK] Base de Conhecimento com 8 regras indexadas e validadas com sucesso!")


=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (PLANTA DE BIODIESEL) ===
ID   | Prioridade | Severidade | SE (Antecedentes)                     | ENTÃO (Consequente)              | Diagnóstico                                                                           | POP                                                                                
-----+------------+------------+---------------------------------------+----------------------------------+---------------------------------------------------------------------------------------+------------------------------------------------------------------------------------
R-01 | 10         | CRÍTICA    | p1 AND t_alta                         | EXOTERMIA_RUNAWAY_REATOR         | Exotermia Descontrolada e Sobrepressão no Reator R-200                                | POP-SIS-01: Desarme total de HT-201, corte de XV-202 e abertura plena de CW-201    
R-02 | 10         | CRÍTICA    | EXOTERMIA_RUNAWAY_REATOR AND v_in_mix | TR

---
## 4. Execução de Simulações e Diagnóstico via Forward Chaining (Data-Driven)

Abaixo são simulados cenários industriais de contingência operacional na planta de biodiesel, demonstrando a inferência em cascata a partir dos sensores discretizados.

In [4]:
motor = MotorInferencia(bc)

# ==============================================================================
# CENÁRIO 1: Runaway Térmico e Trip em Cascata no Reator R-200
# Telemetria: Sobrepressão (p1), Sobretemperatura (t_alta) e Válvula de Metóxido Aberta (v_in_mix)
# ==============================================================================
fatos_c1 = {"p1", "t_alta", "v_in_mix"}
fatos_finais_c1, trilha_c1 = motor.forward_chaining(fatos_c1)

print("=======================================================================")
print("CENÁRIO 1: TRILHA DE INFERÊNCIA FORWARD CHAINING (RUNAWAY REATOR R-200)")
print("=======================================================================")
print(formatar_tabela(trilha_c1))
print(f"\nMemória de Fatos Finais ({len(fatos_finais_c1)}): {sorted(fatos_finais_c1)}")

# ==============================================================================
# CENÁRIO 2: Vazamento de Vapores de Metanol no Setor 100
# Telemetria: Detector de Gás Ativo (g_alm)
# ==============================================================================
fatos_c2 = {"g_alm"}
fatos_finais_c2, trilha_c2 = motor.forward_chaining(fatos_c2)

print("\n=======================================================================")
print("CENÁRIO 2: TRILHA DE INFERÊNCIA FORWARD CHAINING (FUGA DE METANOL SETOR 100)")
print("=======================================================================")
print(formatar_tabela(trilha_c2))

# ==============================================================================
# CENÁRIO 3: Falha de Separação de Fases e Dreno Aberto (Setor 300)
# Telemetria: Válvula de Dreno Aberta (v_glic) e Ausência de Interface (not_i_glic)
# ==============================================================================
fatos_c3 = {"v_glic", "not_i_glic"}
fatos_finais_c3, trilha_c3 = motor.forward_chaining(fatos_c3)

print("\n=======================================================================")
print("CENÁRIO 3: TRILHA DE INFERÊNCIA FORWARD CHAINING (DRENO INDEVIDO SETOR 300)")
print("=======================================================================")
print(formatar_tabela(trilha_c3))


CENÁRIO 1: TRILHA DE INFERÊNCIA FORWARD CHAINING (RUNAWAY REATOR R-200)
Passo | ID Regra | Prioridade | Severidade | Fato Inferido             | Diagnóstico                                                       | Procedimento POP                                                               
------+----------+------------+------------+---------------------------+-------------------------------------------------------------------+--------------------------------------------------------------------------------
1     | R-01     | 10         | CRÍTICA    | EXOTERMIA_RUNAWAY_REATOR  | Exotermia Descontrolada e Sobrepressão no Reator R-200            | POP-SIS-01: Desarme total de HT-201, corte de XV-202 e abertura plena de CW-201
2     | R-02     | 10         | CRÍTICA    | TRIP_ALIMENTACAO_METOXIDO | Corte Imediato da Dosagem de Metóxido por Reação Fora de Controle | POP-SIS-02: Fechar imediatamente XV-202, desenergizar P-102 e inertizar com N2 

Memória de Fatos Finais (5): ['EXOTERMIA_RU

---
## 5. Auditoria Forense e Prova de Metas via Backward Chaining (Goal-Driven / XAI)

No Backward Chaining, o operador ou sistema especialista de segurança funcional interroga o motor a partir de uma hipótese de risco. O motor constrói recursivamente a árvore dedutiva de causa-raiz (*Proof Tree*).

In [5]:
# ==============================================================================
# AUDITORIA 1: Prova Dedutiva da Meta 'TRIP_ALIMENTACAO_METOXIDO'
# Sob o estado de contingência do Cenário 1
# ==============================================================================
meta_1 = "TRIP_ALIMENTACAO_METOXIDO"
sucesso_bc1, trilha_bc1, arvore_bc1 = motor.backward_chaining(meta_1, fatos_c1)

print("=======================================================================")
print(f"AUDITORIA FORENSE 1: BACKWARD CHAINING PARA A META '{meta_1}'")
print("=======================================================================")
for linha in trilha_bc1:
    print(linha)
print(f"\nResultado da Prova Dedutiva: {'META PROVADA (VERDADEIRA)' if sucesso_bc1 else 'FALHA NA PROVA'}")

# ==============================================================================
# AUDITORIA 2: Prova Dedutiva da Meta 'PERDA_BIODIESEL_DRENO_GLICERINA'
# Sob o estado de contingência do Cenário 3
# ==============================================================================
meta_2 = "PERDA_BIODIESEL_DRENO_GLICERINA"
sucesso_bc2, trilha_bc2, arvore_bc2 = motor.backward_chaining(meta_2, fatos_c3)

print("\n=======================================================================")
print(f"AUDITORIA FORENSE 2: BACKWARD CHAINING PARA A META '{meta_2}'")
print("=======================================================================")
for linha in trilha_bc2:
    print(linha)
print(f"\nResultado da Prova Dedutiva: {'META PROVADA (VERDADEIRA)' if sucesso_bc2 else 'FALHA NA PROVA'}")

# ==============================================================================
# AUDITORIA 3: Teste de Hipótese Falsa / Refutação
# Testando se houve 'VAZAMENTO_GAS_METANOL_S100' sob o Cenário 1 (onde apenas o reator falhou)
# ==============================================================================
meta_3 = "VAZAMENTO_GAS_METANOL_S100"
sucesso_bc3, trilha_bc3, arvore_bc3 = motor.backward_chaining(meta_3, fatos_c1)

print("\n=======================================================================")
print(f"AUDITORIA FORENSE 3: PROVA NEGATIVA PARA A META '{meta_3}'")
print("=======================================================================")
for linha in trilha_bc3:
    print(linha)
print(f"\nResultado da Prova Dedutiva: {'META PROVADA' if sucesso_bc3 else 'HIPÓTESE REFUTADA COM SUCESSO (FALSA)'}")


AUDITORIA FORENSE 1: BACKWARD CHAINING PARA A META 'TRIP_ALIMENTACAO_METOXIDO'
[AVALIANDO META] Investigando 'TRIP_ALIMENTACAO_METOXIDO' via 1 regra(s)...
  -> Testando Regra [R-02]: SE (EXOTERMIA_RUNAWAY_REATOR AND v_in_mix) ENTÃO TRIP_ALIMENTACAO_METOXIDO
    [AVALIANDO META] Investigando 'EXOTERMIA_RUNAWAY_REATOR' via 1 regra(s)...
      -> Testando Regra [R-01]: SE (p1 AND t_alta) ENTÃO EXOTERMIA_RUNAWAY_REATOR
        [FATO CONFIRMADO] Hipótese 'p1' presente na base de fatos.
        [FATO CONFIRMADO] Hipótese 't_alta' presente na base de fatos.
      [SUCESSO] Regra [R-01] satisfeita! Meta 'EXOTERMIA_RUNAWAY_REATOR' PROVADA com sucesso.
    [FATO CONFIRMADO] Hipótese 'v_in_mix' presente na base de fatos.
  [SUCESSO] Regra [R-02] satisfeita! Meta 'TRIP_ALIMENTACAO_METOXIDO' PROVADA com sucesso.

Resultado da Prova Dedutiva: META PROVADA (VERDADEIRA)

AUDITORIA FORENSE 2: BACKWARD CHAINING PARA A META 'PERDA_BIODIESEL_DRENO_GLICERINA'
[AVALIANDO META] Investigando 'PERDA_BIODIESEL_

---
## 6. Bateria de Testes Formais e Validação de Asserções (`assert`)

Garantia formal de conformidade dos algoritmos de inferência (Forward & Backward Chaining) e das regras da planta de biodiesel.

In [6]:
# 1. Validação do Catálogo da Base de Conhecimento
assert len(bc.regras) == 8, "A base de conhecimento deve conter exatamente 8 regras cadastradas."
assert len(bc.obter_regras_por_fato("p1")) >= 1, "O índice de fatos deve conter 'p1'."
assert len(bc.obter_regras_por_consequente("TRIP_ALIMENTACAO_METOXIDO")) >= 1, "O índice de consequentes deve conter 'TRIP_ALIMENTACAO_METOXIDO'."

# 2. Validação do Forward Chaining (Cenário 1 - Runaway)
assert "EXOTERMIA_RUNAWAY_REATOR" in fatos_finais_c1, "Cenário 1 deve inferir 'EXOTERMIA_RUNAWAY_REATOR'."
assert "TRIP_ALIMENTACAO_METOXIDO" in fatos_finais_c1, "Cenário 1 deve inferir 'TRIP_ALIMENTACAO_METOXIDO'."

# 3. Validação do Forward Chaining (Cenário 2 - Fuga de Metanol)
assert "VAZAMENTO_GAS_METANOL_S100" in fatos_finais_c2, "Cenário 2 deve inferir 'VAZAMENTO_GAS_METANOL_S100'."

# 4. Validação do Forward Chaining (Cenário 3 - Dreno Glicerina)
assert "PERDA_BIODIESEL_DRENO_GLICERINA" in fatos_finais_c3, "Cenário 3 deve inferir 'PERDA_BIODIESEL_DRENO_GLICERINA'."

# 5. Validação do Backward Chaining
assert sucesso_bc1 is True, "Backward Chaining deve provar 'TRIP_ALIMENTACAO_METOXIDO' sob Cenário 1."
assert sucesso_bc2 is True, "Backward Chaining deve provar 'PERDA_BIODIESEL_DRENO_GLICERINA' sob Cenário 3."
assert sucesso_bc3 is False, "Backward Chaining deve refutar 'VAZAMENTO_GAS_METANOL_S100' sob Cenário 1."

print("[OK] Todos os testes unitários e asserções formais de inferência passaram com 100% de sucesso!")


[OK] Todos os testes unitários e asserções formais de inferência passaram com 100% de sucesso!
